# Домашнее задание 3. Статистическая проверка качества регрессии.


### 1. Загрузите файл candidates.csv с компьютера. 

* откройте первые несколько строк
* изучите строение файла.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import t, f
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# CSV должен лежать рядом с ноутбуком.
file_path = Path("candidates(1).csv")
if not file_path.exists():
    file_path = Path("candidates.csv")

df = pd.read_csv(file_path, sep=";", decimal=",", encoding="utf-8-sig")

display(df.head())
print("Размер таблицы:", df.shape)
df.info()
print("Всего пропусков:", df.isna().sum().sum())
print("Дубликатов строк:", df.duplicated().sum())


,Код претендента,Рейтинг университета окончания,Средний балл диплома,Опыт работы (лет),Возраст,Рейтинг университета диплома,Баллы за сертификаты,Желаемая заработная плата,Итоговая зарплата от компании
0,1001,82,3.22,10,33,84,89,237000,237000
1,1002,4,3.19,3,29,16,0,155000,155000
2,1003,70,3.84,6,30,70,74,212000,197000
3,1004,98,3.67,2,24,97,64,140000,130000
4,1005,94,3.92,2,27,99,66,146000,136000


Размер таблицы: (800, 9)
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Код претендента                 800 non-null    int64  
 1   Рейтинг университета окончания  800 non-null    int64  
 2   Средний балл диплома            800 non-null    float64
 3   Опыт работы (лет)               800 non-null    int64  
 4   Возраст                         800 non-null    int64  
 5   Рейтинг университета диплома    800 non-null    int64  
 6   Баллы за сертификаты            800 non-null    int64  
 7   Желаемая заработная плата       800 non-null    int64  
 8   Итоговая зарплата от компании   800 non-null    int64  
dtypes: float64(1), int64(8)
memory usage: 56.4 KB
Всего пропусков: 0
Дубликатов строк: 0


### 2. На основе файла создайте матрицу X с характеристиками (некоторые столбцы могут не понадобятся для характеристик, подумайте какие), и столбец y (из последнего столбца).

* отобразите первые строки матрицы X и столбца y на экране.


In [2]:
# Первый столбец — идентификатор, последний — целевая переменная.
X = df.iloc[:, 1:-1].copy()
y = df.iloc[:, -1].copy()

# Сохраняем исходные признаки для сравнения корреляций в задании 7.
X_all = X.copy()

print("Матрица X:")
display(X.head())
print("Целевая переменная y:")
display(y.head().to_frame())
print("Число наблюдений:", len(y))
print("Число признаков:", X.shape[1])


Матрица X:


,Рейтинг университета окончания,Средний балл диплома,Опыт работы (лет),Возраст,Рейтинг университета диплома,Баллы за сертификаты,Желаемая заработная плата
0,82,3.22,10,33,84,89,237000
1,4,3.19,3,29,16,0,155000
2,70,3.84,6,30,70,74,212000
3,98,3.67,2,24,97,64,140000
4,94,3.92,2,27,99,66,146000


Целевая переменная y:


,Итоговая зарплата от компании
0,237000
1,155000
2,197000
3,130000
4,136000


Число наблюдений: 800
Число признаков: 7


### 3. Используя SAG, постройте на этих данных модель линейной регрессии

* выведите на экран полученное уравнение линиии


In [3]:
def fit_sag(X_train, y_train):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    # alpha=0 отключает регуляризацию: получаем обычную линейную регрессию.
    model = Ridge(alpha=0, solver="sag", max_iter=100000,
                  tol=1e-10, random_state=42)
    model.fit(X_scaled, y_train)
    return model, scaler


def print_equation(intercept, coefficients, feature_names):
    print(f"y_hat = {intercept:.6f}")
    for coefficient, name in zip(coefficients, feature_names):
        print(f"    {coefficient:+.6f} * [{name}]")


model, scaler = fit_sag(X, y)
y_pred = model.predict(scaler.transform(X))

# Возвращаем коэффициенты из стандартизированных в исходные единицы.
coefficients = model.coef_ / scaler.scale_
intercept = model.intercept_ - coefficients @ scaler.mean_
weights = np.r_[intercept, coefficients]

print_equation(intercept, coefficients, X.columns)
print("Итераций SAG:", model.n_iter_[0])


y_hat = 36695.887725
    -154.744816 * [Рейтинг университета окончания]
    +11669.838403 * [Средний балл диплома]
    +9648.406166 * [Опыт работы (лет)]
    +64.994720 * [Возраст]
    -199.401562 * [Рейтинг университета диплома]
    +230.716352 * [Баллы за сертификаты]
    +0.326216 * [Желаемая заработная плата]
Итераций SAG: 82


### 4. Проверьте веса на значимость, используя алгоритм и формулы из лекции

* код должен соотвествовать алгоритму из лекции
* объясните логически, почему мы проверяем две гипотезы
* какие веса у вас оказались незначимыми?


In [4]:
alpha_sig = 0.05  # Уровень значимости; не путать с alpha у Ridge.
n, k = X.shape
degrees = n - k - 1

# Формулы со страниц 1–2 лекции.
X_const = np.column_stack((np.ones(n), X.to_numpy()))
s2 = np.sum((y - y_pred) ** 2) / degrees
C = s2 * np.linalg.inv(X_const.T @ X_const)
standard_errors = np.sqrt(np.diag(C))
t_values = weights / standard_errors
t_critical = t.ppf(1 - alpha_sig / 2, degrees)
p_values = 2 * t.sf(np.abs(t_values), degrees)

weight_test = pd.DataFrame({
    "Вес": weights,
    "Стандартная ошибка": standard_errors,
    "t": t_values,
    "p-value": p_values,
    "Значим": np.abs(t_values) > t_critical
}, index=["Свободный член"] + list(X.columns))

display(weight_test)
print("Степени свободы:", degrees)
print(f"Интервал неотвержения H0: [{-t_critical:.6f}; {t_critical:.6f}]")

# Отбираем столбцы X; свободный член не является отдельным признаком.
insignificant = list(X.columns[np.abs(t_values[1:]) <= t_critical])
print("Незначимые признаки:", insignificant)


,Вес,Стандартная ошибка,t,p-value,Значим
Свободный член,36695.887725,5893.967199,6.226008,7.755663e-10,True
Рейтинг университета окончания,-154.744816,43.050605,-3.594486,3.451405e-04,True
Средний балл диплома,11669.838403,589.661594,19.790738,3.862332e-71,True
Опыт работы (лет),9648.406166,308.520856,31.273108,1.896109e-140,True
Возраст,64.994720,217.693375,0.298561,7.653535e-01,False
Рейтинг университета диплома,-199.401562,42.266715,-4.717697,2.819168e-06,True
Баллы за сертификаты,230.716352,11.016895,20.942049,7.738177e-78,True
Желаемая заработная плата,0.326216,0.012699,25.688524,2.461603e-106,True


Степени свободы: 792
Интервал неотвержения H0: [-1.962964; 1.962964]
Незначимые признаки: ['Возраст']


**Почему мы проверяем две гипотезы?**

Мы проверяем две гипотезы, чтобы понять, действительно ли конкретная характеристика влияет на результат модели или полученный вес мог появиться случайно. Первая гипотеза предполагает, что характеристика не вносит самостоятельного вклада в предсказание, а вторая — что она действительно важна. Так мы можем определить, какие признаки стоит оставить в модели, а какие можно убрать.

**Какие веса оказались незначимыми?**

Незначимым оказался только вес возраста. Это означает, что при наличии остальных характеристик возраст не даёт модели достаточно новой информации. Поэтому его можно исключить. Все остальные веса оказались значимыми.

### 5. Если какие-то веса оказались незначимыми, перепишите матрицу характеристик X без соответствующих столбцов, обучите заново модель, используя SAG


In [5]:
X = X.drop(columns=insignificant)

model, scaler = fit_sag(X, y)
y_pred = model.predict(scaler.transform(X))
coefficients = model.coef_ / scaler.scale_
intercept = model.intercept_ - coefficients @ scaler.mean_

print("Оставшиеся признаки:")
for name in X.columns:
    print(name)
print("Размер новой матрицы X:", X.shape)
print_equation(intercept, coefficients, X.columns)
print("Итераций SAG:", model.n_iter_[0])


Оставшиеся признаки:
Рейтинг университета окончания
Средний балл диплома
Опыт работы (лет)
Рейтинг университета диплома
Баллы за сертификаты
Желаемая заработная плата
Размер новой матрицы X: (800, 6)
y_hat = 38290.304936
    -154.283835 * [Рейтинг университета окончания]
    +11670.337679 * [Средний балл диплома]
    +9714.973238 * [Опыт работы (лет)]
    -199.828830 * [Рейтинг университета диплома]
    +230.644493 * [Баллы за сертификаты]
    +0.326051 * [Желаемая заработная плата]
Итераций SAG: 81


**Ответ к заданию 5.** Из `X` удалён **«Возраст»**, затем модель заново обучена с помощью SAG. В итоговой матрице **6 признаков**, её размер — **800 × 6**. Удаление выполнено по результату проверки, а не по заранее заданному имени столбца.

Обозначим признаки:

| Обозначение | Признак |
|---|---|
| $x_1$ | Рейтинг университета окончания |
| $x_2$ | Средний балл диплома |
| $x_3$ | Опыт работы (лет) |
| $x_4$ | Рейтинг университета диплома |
| $x_5$ | Баллы за сертификаты |
| $x_6$ | Желаемая заработная плата |

Итоговое уравнение с округлёнными коэффициентами:

$$\begin{aligned}
\hat y={}&38290{,}304936-154{,}283835x_1+11670{,}337679x_2\\
&+9714{,}973238x_3-199{,}828830x_4\\
&+230{,}644493x_5+0{,}326050822x_6.
\end{aligned}$$

Например, при увеличении опыта на один год и фиксированных остальных признаках прогноз повышается примерно на **9714,97** денежных единиц. Это интерпретация коэффициента модели, а не доказанный причинный эффект. Отрицательные коэффициенты рейтингов относятся к их **числовым значениям**; направление шкал качества вузов в файле не объяснено.


### 6. Проверьте модель на значимость по алгоритму из лекции

* код должен соответствовать алгоритму из лекции
* объясните логически, почему мы проверяем две гипотезы и что эти гипотезы собой подразумевают
* какая у вас получилась модель?
* подходит ли она для описания наших данных?


In [6]:
n, k = X.shape
degrees = n - k - 1

# Сначала считаем R², поскольку он входит в формулу F из лекции.
SSE = np.sum((y - y_pred) ** 2)
SST = np.sum((y - y.mean()) ** 2)
R2 = 1 - SSE / SST

F_value = (R2 / k) / ((1 - R2) / degrees)
F_critical = f.ppf(1 - alpha_sig, k, degrees)
p_F = f.sf(F_value, k, degrees)

print(f"n = {n}, k = {k}, степени свободы F: ({k}, {degrees})")
print(f"F = {F_value:.6f}")
print(f"F критическое = {F_critical:.6f}")
if p_F < 0.001:
    print("p-value < 0.001")
else:
    print(f"p-value = {p_F:.6f}")

if F_value > F_critical:
    print("H0 отвергается: модель статистически значима.")
else:
    print("Нет оснований отвергнуть H0 о незначимости модели.")


n = 800, k = 6, степени свободы F: (6, 793)
F = 9183.214570
F критическое = 2.109995
p-value < 0.001
H0 отвергается: модель статистически значима.


**Почему мы проверяем две гипотезы и что они подразумевают?**

Здесь мы проверяем уже не отдельные характеристики, а всю модель целиком. Первая гипотеза предполагает, что все признаки бесполезны и модель не даёт преимуществ перед обычным средним значением. Вторая гипотеза говорит, что хотя бы один признак действительно помогает предсказывать зарплату. Такая проверка позволяет понять, есть ли вообще смысл использовать построенную модель.

**Какая получилась модель?**

Получилась статистически значимая линейная модель, которая использует шесть характеристик кандидатов для предсказания зарплаты. Возраст мы исключили, поскольку он оказался незначимым.

**Подходит ли она для описания наших данных?**

Да, модель хорошо подходит для описания наших данных. Проверка показала, что связь между характеристиками кандидатов и зарплатой не случайна. При этом сама по себе статистическая значимость ещё не означает, что модель предсказывает зарплату без ошибок.

### 7. Рассчитайте парный коэффициент корреляции для каждой пары x_i, y, используя алгоритм и формулы из лекции.

* код должен соотвествовать алгоритму из лекции
* ответьте на вопрос "что в общем случае (а не в конкретно нашем) показывают парные коэффициенты корреляции?"
* выведите коэффициент корреляции на экран для каждой пары в виде: пара, значение коэффициента
* ответьте на вопрос "что полученные значения говорят о нашей модели?"
* какие характеристики оказывают на ответ наибольшее значение, какие наименьшее?


In [7]:
# Рассчитываем корреляции всех исходных признаков, включая удалённый возраст.
y_centered = y - y.mean()
sigma_y = np.sqrt(np.mean(y_centered ** 2))
correlations = {}

for name in X_all.columns:
    x = X_all[name]
    x_centered = x - x.mean()
    sigma_x = np.sqrt(np.mean(x_centered ** 2))
    r = np.mean(x_centered * y_centered) / (sigma_x * sigma_y)
    correlations[name] = r
    print(f"({name}, y): {r:.6f}")


(Рейтинг университета окончания, y): -0.272495
(Средний балл диплома, y): 0.137158
(Опыт работы (лет), y): 0.941768
(Возраст, y): 0.890657
(Рейтинг университета диплома, y): -0.258448
(Баллы за сертификаты, y): 0.171400
(Желаемая заработная плата, y): 0.973753


**Что в общем случае показывают парные коэффициенты корреляции?**

Парные коэффициенты корреляции показывают, насколько сильно и в каком направлении связаны две характеристики. Положительная корреляция означает, что при увеличении одной величины другая обычно тоже растёт, а отрицательная — что она обычно уменьшается. Чем ближе коэффициент к нулю, тем слабее линейная связь. При этом корреляция не обязательно означает, что одна характеристика является причиной изменения другой.

**Что полученные значения говорят о нашей модели?**

Полученные значения показывают, что некоторые характеристики кандидатов действительно сильно связаны с итоговой зарплатой, а другие — гораздо слабее. Это помогает понять, на каких признаках модель может строить свои прогнозы. При этом сильная корреляция не всегда означает, что признак обязательно нужен в модели, поскольку разные характеристики могут дублировать информацию друг друга.

**Какие характеристики оказывают наибольшее и наименьшее значение на ответ?**

Самая сильная связь с зарплатой наблюдается у желаемой зарплаты и опыта работы. Возраст тоже сильно связан с зарплатой, но после учёта других признаков его самостоятельный вклад оказался незначимым. Самая слабая парная связь наблюдается у среднего балла диплома и баллов за сертификаты. Однако это не означает, что они совсем бесполезны для модели.

### 8. Вычислите коэффициент детерминации

* код должен соответствовать формуле из лекции
* выведите полученный коэффициент на экран
* что он говорит о нашей модели? насколько качественно она описала данные? лучше или хуже она, чем средняя линия?


In [8]:
# Формула со страниц 2–3 лекции.
SSE = np.sum((y - y_pred) ** 2)
SST = np.sum((y - y.mean()) ** 2)
R2 = 1 - SSE / SST

# Для интерпретации сравним ошибки в исходных единицах зарплаты.
rmse_train = np.sqrt(SSE / len(y))
rmse_mean = np.sqrt(SST / len(y))

print(f"R² = {R2:.9f}")
print(f"Объяснённая доля вариации: {R2:.4%}")
print(f"RMSE модели на обучении: {rmse_train:.2f}")
print(f"RMSE средней линии на этих данных: {rmse_mean:.2f}")


R² = 0.985811996
Объяснённая доля вариации: 98.5812%
RMSE модели на обучении: 8655.16
RMSE средней линии на этих данных: 72663.17


**Что коэффициент детерминации говорит о нашей модели? Насколько качественно она описывает данные? Лучше или хуже средней линии?**

Коэффициент детерминации получился очень близким к единице. Это означает, что наша модель объясняет почти 99% различий в зарплатах кандидатов на имеющихся данных. Следовательно, она очень хорошо описывает зависимость зарплаты от выбранных характеристик.

По сравнению со средней линией наша модель работает значительно лучше, поскольку учитывает особенности каждого кандидата, а не просто предсказывает всем одинаковую среднюю зарплату.

### 9. Рассчитайте LOOCV кросс-валидации, используя алгоритм и формулы из лекции.

* код должен соотвествовать алгоритму из лекции
* объясните устно как он работает и зачем он нужен
* выведите LOOCV на экран
* ответьте на вопрос "что полученный LOOCV говорит о нашей модели? она идеально нам подходит, или в ней нужно что-то менять?"


In [9]:
n = len(y)
errors = []
mean_errors = []

for i in range(n):
    # Одна строка — test, остальные n - 1 строк — train.
    train_mask = np.arange(n) != i
    X_train = X.iloc[train_mask]
    y_train = y.iloc[train_mask]
    X_test = X.iloc[[i]]

    # Новый StandardScaler обучается только на train внутри fit_sag.
    model_i, scaler_i = fit_sag(X_train, y_train)
    prediction = model_i.predict(scaler_i.transform(X_test))[0]
    errors.append(y.iloc[i] - prediction)

    # Базовая модель предсказывает среднее только обучающей части.
    mean_errors.append(y.iloc[i] - y_train.mean())

# По лекции LOOCV — корень из среднего квадрата ошибок (RMSE).
LOOCV = np.sqrt(np.mean(np.array(errors) ** 2))
LOOCV_mean = np.sqrt(np.mean(np.array(mean_errors) ** 2))

print("Количество проверенных наблюдений:", len(errors))
print(f"LOOCV модели (RMSE): {LOOCV:.2f}")
print(f"LOOCV средней линии (RMSE): {LOOCV_mean:.2f}")
print(f"RMSE модели на обучении: {rmse_train:.2f}")
print(f"Средняя зарплата: {y.mean():.2f}")
print(f"LOOCV / средняя зарплата: {LOOCV / y.mean():.2%}")


Количество проверенных наблюдений: 800
LOOCV модели (RMSE): 8739.41
LOOCV средней линии (RMSE): 72754.11
RMSE модели на обучении: 8655.16
Средняя зарплата: 229032.50
LOOCV / средняя зарплата: 3.82%


**Что полученный LOOCV говорит о нашей модели? Она идеально нам подходит или в ней нужно что-то менять?**

LOOCV позволяет проверить, насколько хорошо модель предсказывает зарплату кандидата, данные которого не использовались при её обучении. Для этого мы по очереди исключаем одного кандидата, обучаем модель на остальных и проверяем, насколько точно она предскажет его зарплату.

Полученная ошибка составляет около 9 тысяч, что относительно невелико по сравнению со средней зарплатой в наших данных. Кроме того, ошибка при проверке почти не отличается от ошибки на обучающих данных. Это говорит о том, что модель достаточно хорошо справляется с прогнозированием и явных признаков сильного переобучения нет.